### Create inference and pkl for the training feature

In [3]:
# Import Libraries
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import pickle
import tarfile
import joblib
import boto3
import os
import sys
import shutil
from datetime import datetime
import sagemaker
import google.protobuf
from sagemaker.sklearn.model import SKLearnModel
from sagemaker import Session
from sagemaker import get_execution_role, Session

In [4]:
# bash to create new enviroment
# Create a clean Conda env with Python 3.7
#conda create -n cardio_env python=3.7 -y

# Activate the environment (start a new terminal if needed)
#source ~/.bashrc  # Apply changes
#conda activate cardio_env

#pip install pandas==1.0.5 \
            #scikit-learn==0.23.2 \
            #joblib==0.14.1 \
            #boto3==1.17.106 \
            #sagemaker==2.34.0 \
            #protobuf==3.12.0

In [5]:
# Correct Dependencies
# Python 3.7.12
# boto3 1.17.106
# joblib 0.14.1
# pandas 1.0.5
# protobuf 3.12.0
# sagemaker 2.34.0
# scikit-learn 0.23.2

# Check Dependencies
print("Python:", sys.version)
print("boto3:", boto3.__version__)
print("joblib:", joblib.__version__)
print("pandas:", pd.__version__)
print("protobuf:", google.protobuf.__version__)
print("sagemaker:", sagemaker.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.7.12 | packaged by conda-forge | (default, Oct 26 2021, 06:08:21) 
[GCC 9.4.0]
boto3: 1.17.106
joblib: 0.14.1
pandas: 1.0.5
protobuf: 3.12.0
sagemaker: 2.34.0
scikit-learn: 0.23.2


In [6]:
# Initialize SageMaker session and role
sagemaker_session = Session()
role = get_execution_role()
bucket = "sagemaker-us-east-1-226675648827"

# Feature columns
FEATURE_COLUMNS = [
    'age', 'gender', 'height_ft', 'weight_lbs', 'systolic_bp', 'diastolic_bp',
    'cholesterol', 'gluc', 'smoke', 'alco', 'active',
    'bmi', 'age_group', 'cholesterol_label', 'pulse_pressure', 'chol_bmi_ratio',
    'height_in', 'age_years', 'is_hypertensive', 'bp_category', 'bmi_category',
    'age_gluc_interaction', 'lifestyle_score'
]

# Load training data
train_df = pd.read_csv(f"s3://{bucket}/cardio_data/cardio_prod_split40.csv")
X = train_df.drop(columns=["cardio"])
y = train_df["cardio"]
train_df.head()

,age,height_ft,weight_lbs,systolic_bp,diastolic_bp,cholesterol,gluc,smoke,alco,active,...,age_years,is_hypertensive,age_gluc_interaction,lifestyle_score,gender,bp_category,bmi_category,age_group,cholesterol_label,cardio
0,-0.860483,-0.323580,-0.914107,-0.410657,-0.147087,-0.538657,-0.390761,-0.312731,-0.23822,0.494625,...,-0.860483,-0.607947,-0.543807,-0.596864,0.0,2.0,0.0,1.0,1.0,0
1,-0.268438,1.211180,0.266123,-0.410657,-0.147087,-0.538657,-0.390761,-0.312731,-0.23822,-2.021734,...,-0.268438,-0.607947,-0.422052,1.161966,1.0,2.0,2.0,2.0,1.0,0
2,0.915651,-1.052592,-0.983384,-0.410657,-0.147087,-0.538657,-0.390761,-0.312731,-0.23822,0.494625,...,0.915651,-0.607947,-0.178543,-0.596864,0.0,2.0,0.0,2.0,1.0,1
3,-0.712472,-0.822378,-1.191845,-0.410657,-0.147087,-0.538657,-0.390761,-0.312731,-0.23822,0.494625,...,-0.712472,-0.607947,-0.513368,-0.596864,0.0,2.0,0.0,1.0,1.0,0
4,0.323606,-0.170104,-0.358631,-0.410657,-0.147087,-0.538657,-0.390761,-0.312731,-0.23822,0.494625,...,0.323606,-0.607947,-0.300298,-0.596864,0.0,2.0,2.0,2.0,1.0,0


In [7]:
# Train Logistic Regression model
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X, y)
joblib.dump(log_model, "logistic_model.pkl", protocol=4)

['logistic_model.pkl']

In [8]:
# Train Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=25,
    min_samples_split=7,
    min_samples_leaf=3,
    bootstrap=True,
    random_state=42
)
rf_model.fit(X, y)
joblib.dump(rf_model, "random_forest_model.pkl", protocol=4)

['random_forest_model.pkl']

In [9]:
# Write inference.py for logistic model
with open("inference.py", "w") as f:
    f.write(f'''
import pandas as pd
from io import StringIO
import joblib
import os

FEATURE_COLUMNS = {FEATURE_COLUMNS}

def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, "logistic_model.pkl"))

def input_fn(input_data, content_type):
    if content_type == "text/csv":
        df = pd.read_csv(StringIO(input_data), header=None)
        if df.shape[1] != len(FEATURE_COLUMNS):
            raise ValueError(f"Expected {{len(FEATURE_COLUMNS)}} features, got {{df.shape[1]}}")
        df.columns = FEATURE_COLUMNS
        return df
    else:
        raise ValueError("Unsupported content type: " + content_type)

def predict_fn(input_data, model):
    return model.predict(input_data)

def output_fn(prediction, content_type):
    return '\\n'.join(str(x) for x in prediction)
''')

In [10]:
# Write inference_rf.py for random forest
with open("inference_rf.py", "w") as f:
    f.write(f'''
import pandas as pd
from io import StringIO
import joblib
import os

FEATURE_COLUMNS = {FEATURE_COLUMNS}

def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, "random_forest_model.pkl"))

def input_fn(input_data, content_type):
    if content_type == "text/csv":
        df = pd.read_csv(StringIO(input_data), header=None)
        if df.shape[1] != len(FEATURE_COLUMNS):
            raise ValueError(f"Expected {{len(FEATURE_COLUMNS)}} features, got {{df.shape[1]}}")
        df.columns = FEATURE_COLUMNS
        return df
    else:
        raise ValueError("Unsupported content type: " + content_type)

def predict_fn(input_data, model):
    return model.predict(input_data)

def output_fn(prediction, content_type):
    return '\\n'.join(str(x) for x in prediction)
''')

In [11]:
# Package models and inference scripts
with tarfile.open("logistic_model.tar.gz", "w:gz") as tar:
    tar.add("logistic_model.pkl")
    tar.add("inference.py")

with tarfile.open("random_forest_model.tar.gz", "w:gz") as tar:
    tar.add("random_forest_model.pkl")
    tar.add("inference_rf.py")

# Upload to S3
s3 = boto3.client("s3")
bucket = "sagemaker-us-east-1-226675648827"
s3.upload_file("logistic_model.tar.gz", bucket, "model/logistic/logistic_model.tar.gz")
s3.upload_file("random_forest_model.tar.gz", bucket, "model/random_forest/random_forest_model.tar.gz")

print("Models resaved with protocol=4 and uploaded to S3.")

Models resaved with protocol=4 and uploaded to S3.


In [12]:
# Batch Transform for Logistic Regression
logistic_model = SKLearnModel(
    model_data=f"s3://{bucket}/model/logistic/logistic_model.tar.gz",
    role=role,
    entry_point="inference.py",
    framework_version="0.23-1",
    sagemaker_session=sagemaker_session
)
logistic_transformer = logistic_model.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/batch_output/logistic/",
    assemble_with="Line",
    accept="text/csv"
)
logistic_transformer.transform(
    data=f"s3://{bucket}/cardio_data/cardio_prod_no_label_v2.csv",
    content_type="text/csv",
    split_type="Line",
    input_filter="$[0:]",
    join_source="Input",
    output_filter="$[0,-1]",
    wait=True
)
print("Logistic Regression batch transform complete.")

..............................
2025-06-15 22:36:01,355 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2025-06-15 22:36:01,359 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2025-06-15 22:36:01,360 INFO - sagemaker-containers - nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
    keepalive_timeout 3;
    location ~ ^/(ping|invocations|execution-parameters) {
      proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
      proxy_set_header Host $http_host;
      proxy_redirect off;
      proxy_read_timeout 60s;
      proxy_pass http://gunicorn;
    }
    location / {
      r

In [13]:
# Batch Transform for Random Forest
rf_model = SKLearnModel(
    model_data=f"s3://{bucket}/model/random_forest/random_forest_model.tar.gz",
    role=role,
    entry_point="inference_rf.py",
    framework_version="0.23-1",
    sagemaker_session=sagemaker_session
)
rf_transformer = rf_model.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/batch_output/random_forest/",
    assemble_with="Line",
    accept="text/csv"
)
rf_transformer.transform(
    data=f"s3://{bucket}/cardio_data/cardio_prod_no_label_v2.csv",
    content_type="text/csv",
    split_type="Line",
    input_filter="$[0:]",
    join_source="Input",
    output_filter="$[0,-1]",
    wait=True
)
print("Random Forest batch transform complete.")

...........................2025-06-15 22:41:10,497 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2025-06-15 22:41:10,501 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2025-06-15 22:41:10,502 INFO - sagemaker-containers - nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
    keepalive_timeout 3;
    location ~ ^/(ping|invocations|execution-parameters) {
      proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
      proxy_set_header Host $http_host;
      proxy_redirect off;
      proxy_read_timeout 60s;
      proxy_pass http://gunicorn;
    }
    location / {
      retur

In [14]:
# Save feature group to S3
!aws s3 cp cardio_inference_transform_both_models.ipynb s3://sagemaker-us-east-1-226675648827/cardio-project/cardio_inference_transform_both_models.ipynb

upload: ./cardio_inference_transform_both_models.ipynb to s3://sagemaker-us-east-1-226675648827/cardio-project/cardio_inference_transform_both_models.ipynb
